# Fantasy Premier League Data Collection via API

This notebook demonstrates how to fetch, process, and merge Fantasy Premier League (FPL) data using the official FPL API. The goal is to create a comprehensive dataset that includes:
- Player performance statistics for each gameweek
- Player attributes (cost, form, ICT index)
- Team strength metrics (attack, defense, home/away ratings)
- Fixture information (opponent, difficulty, home/away status)

This enriched dataset can be used for predictive modeling and analysis of FPL points.

In [ ]:
import requests
import pandas as pd

## Step 1: Fetch Raw Gameweek Data

We'll start by collecting player performance data from the FPL API for all 38 gameweeks of the season. The `/event/{gw}/live/` endpoint provides detailed statistics for each player in a specific gameweek, including:
- Minutes played
- Goals scored
- Assists
- Bonus points
- Expected goals (xG) and expected assists (xA)
- Clean sheets, saves, penalties, cards, etc.

Each player's data will be tagged with their gameweek number for temporal tracking.

In [ ]:
# Initialize list to store all player-gameweek records
all_gw_data = []

# Loop through all 38 gameweeks of the Premier League season
for gw in range(1, 39):   # up to GW38
    # Construct API URL for the specific gameweek
    url = f"https://fantasy.premierleague.com/api/event/{gw}/live/"
    
    # Make GET request and parse JSON response
    data = requests.get(url).json()
    
    # Extract player data from 'elements' key
    for p in data['elements']:
        # Tag each player record with the gameweek number for temporal tracking
        p['gw'] = gw
        all_gw_data.append(p)

# Convert the list of dictionaries to a pandas DataFrame for easier manipulation
stats_df = pd.DataFrame(all_gw_data)

# Save raw data to CSV for backup/reference
stats_df.to_csv("fpl_2025_26.csv", index=False)

# Display dataset information
print("Raw FPL data fetched. Shape:", stats_df.shape)
print(stats_df.head())

Raw FPL data fetched. Shape: (12619, 5)
   id                                              stats  \
0   1  {'minutes': 90, 'goals_scored': 0, 'assists': ...   
1   2  {'minutes': 0, 'goals_scored': 0, 'assists': 0...   
2   3  {'minutes': 0, 'goals_scored': 0, 'assists': 0...   
3   4  {'minutes': 0, 'goals_scored': 0, 'assists': 0...   
4   5  {'minutes': 90, 'goals_scored': 0, 'assists': ...   

                                             explain  modified  gw  
0  [{'fixture': 9, 'stats': [{'identifier': 'minu...     False   1  
1  [{'fixture': 9, 'stats': [{'identifier': 'minu...     False   1  
2  [{'fixture': 9, 'stats': [{'identifier': 'minu...     False   1  
3  [{'fixture': 9, 'stats': [{'identifier': 'minu...     False   1  
4  [{'fixture': 9, 'stats': [{'identifier': 'minu...     False   1  


## Step 2: Expand Nested Statistics

The raw data contains a `stats` column that holds a nested dictionary with all the detailed performance metrics. We need to flatten this structure to make each statistic its own column, which will allow us to:
- Easily access individual statistics
- Perform calculations and aggregations
- Use the data for machine learning models

This expansion transforms data from:
```
stats: {minutes: 90, goals_scored: 2, assists: 1, ...}
```
To separate columns:
```
minutes: 90, goals_scored: 2, assists: 1, ...
```

In [ ]:
# The 'stats' column contains a nested dictionary of all player performance metrics for the gameweek
# We use apply(pd.Series) to expand each dictionary into separate columns
stats_expanded = stats_df['stats'].apply(pd.Series)

# Preserve critical identifiers: player ID and gameweek number
# These are needed to maintain relationships with other datasets
stats_expanded['id'] = stats_df['id']
stats_expanded['gw'] = stats_df['gw']

# Store the cleaned, expanded DataFrame
df_clean = stats_expanded

# Display information about the expanded dataset
print("Expanded FPL stats dataframe. Shape:", df_clean.shape)
print(df_clean.head())
print("Available columns:", df_clean.columns.tolist())

Expanded FPL stats dataframe. Shape: (8818, 30)
   minutes  goals_scored  assists  clean_sheets  goals_conceded  own_goals  \
0       90             0        0             1               0          0   
1        0             0        0             0               0          0   
2        0             0        0             0               0          0   
3        0             0        0             0               0          0   
4       90             0        0             1               0          0   

   penalties_saved  penalties_missed  yellow_cards  red_cards  ...  \
0                0                 0             1          0  ...   
1                0                 0             0          0  ...   
2                0                 0             0          0  ...   
3                0                 0             0          0  ...   
4                0                 0             0          0  ...   

   defensive_contribution  starts  expected_goals expected_ass

## Step 3: Fetch Supplementary Data

While we now have detailed gameweek statistics, we need additional contextual information to build a comprehensive dataset:

### **A. Bootstrap-Static Data**
This endpoint provides static information about players and teams:
- **Player attributes**: Name, team, position, cost, form, ICT index (Influence, Creativity, Threat)
- **Team metrics**: Strength ratings for attack/defense in home/away scenarios

### **B. Fixtures Data**
This endpoint provides match-level information:
- Which teams are playing each other
- Fixture difficulty ratings
- Home/away designations
- Kickoff times

These datasets will be merged with our gameweek statistics to add valuable features for prediction models.

In [ ]:
import requests
import pandas as pd

# 1️⃣ Bootstrap Data (Players and Teams)
bootstrap_url = "https://fantasy.premierleague.com/api/bootstrap-static/"
bootstrap_data = requests.get(bootstrap_url).json()

# Extract player information
players_df = pd.DataFrame(bootstrap_data['elements'])
# Select only relevant player-level attributes to avoid data bloat
players_df = players_df[[
    'id',                              # Unique player identifier
    'first_name', 'second_name',       # Player names
    'team',                            # Team ID
    'element_type',                    # Position (1=GK, 2=DEF, 3=MID, 4=FWD)
    'now_cost',                        # Current player cost (in 0.1m units)
    'chance_of_playing_next_round',    # Injury/availability status (0-100%)
    'form',                            # Recent form score
    'influence', 'creativity', 'threat', 'ict_index'  # Performance metrics
]]

# Extract team information
teams_df = pd.DataFrame(bootstrap_data['teams'])
teams_df = teams_df[[
    'id',                              # Team identifier
    'strength_overall_home',           # Overall team strength at home
    'strength_overall_away',           # Overall team strength away
    'strength_attack_home',            # Offensive rating at home
    'strength_attack_away',            # Offensive rating away
    'strength_defence_home',           # Defensive rating at home
    'strength_defence_away',           # Defensive rating away
    'form'                             # Recent team form
]]

# 2️⃣ Fixtures Data
fixtures_url = "https://fantasy.premierleague.com/api/fixtures/"
fixtures_df = pd.DataFrame(requests.get(fixtures_url).json())
# Select relevant fixture-level columns
fixtures_df = fixtures_df[[
    'id',                              # Unique fixture identifier
    'event',                           # Gameweek number
    'team_h',                          # Home team ID
    'team_a',                          # Away team ID
    'kickoff_time',                    # Match kickoff timestamp
    'minutes',                         # Minutes elapsed in match
    'team_h_difficulty',               # Difficulty rating for home team
    'team_a_difficulty'                # Difficulty rating for away team
]]

# Display summary information about the fetched datasets
print("Players shape:", players_df.shape)
print("Teams shape:", teams_df.shape)
print("Fixtures shape:", fixtures_df.shape)
print("Players columns:", players_df.columns.tolist())
print("Teams columns:", teams_df.columns.tolist())
print("Fixtures columns:", fixtures_df.columns.tolist())

Players shape: (755, 12)
Teams shape: (20, 8)
Fixtures shape: (380, 8)
players cols: Index(['id', 'first_name', 'second_name', 'team', 'element_type', 'now_cost',
       'chance_of_playing_next_round', 'form', 'influence', 'creativity',
       'threat', 'ict_index'],
      dtype='object')
teams cols: Index(['id', 'strength_overall_home', 'strength_overall_away',
       'strength_attack_home', 'strength_attack_away', 'strength_defence_home',
       'strength_defence_away', 'form'],
      dtype='object')
fixtures cols: Index(['id', 'event', 'team_h', 'team_a', 'kickoff_time', 'minutes',
       'team_h_difficulty', 'team_a_difficulty'],
      dtype='object')


## Step 4: Merge All Data Sources

This is the most critical step where we combine all the data sources into a unified dataset. The merging process involves several challenges:

### **Challenge: Home vs Away Players**
Each fixture has players from both teams, but they face different opponents and play at different venues. We need to handle this by:
1. Creating separate records for home players (with away team as opponent)
2. Creating separate records for away players (with home team as opponent)
3. Combining both sets while maintaining proper opponent and difficulty information

### **Merging Strategy**
1. **Add player attributes** → Enriches with names, cost, form, ICT
2. **Merge home fixtures** → Links home players to their fixtures
3. **Merge away fixtures** → Links away players to their fixtures
4. **Concatenate** → Combines home and away records
5. **Add team strength** → Enriches with defensive/offensive ratings

The final dataset will have one row per player per gameweek with complete context about their match situation.

In [ ]:
# 1️⃣ Add Player Attributes
# Merge player information (names, team, cost, form, ICT) to each gameweek record
df_with_player = df_clean.merge(
    players_df,
    on='id',           # Join on player ID
    how='left'         # Keep all gameweek records even if player info is missing
)

print("Columns after merging player info:", df_with_player.columns.tolist())

# 2️⃣ Merge Home Fixtures
# Link players to fixtures where their team was playing at home
df_home = df_with_player.merge(
    fixtures_df,
    left_on=['gw', 'team'],      # Match on gameweek and team ID
    right_on=['event', 'team_h'], # From fixtures, match event and home team
    how='left'                    # Keep all player records
)
# Add fixture context for home players
df_home['is_home'] = 1                              # Flag: playing at home
df_home['opponent_team_id'] = df_home['team_a']     # Opponent is the away team
df_home['opponent_difficulty'] = df_home['team_h_difficulty']  # Use home difficulty

# 3️⃣ Merge Away Fixtures
# Link players to fixtures where their team was playing away
df_away = df_with_player.merge(
    fixtures_df,
    left_on=['gw', 'team'],      # Match on gameweek and team ID
    right_on=['event', 'team_a'], # From fixtures, match event and away team
    how='left'                    # Keep all player records
)
# Add fixture context for away players
df_away['is_home'] = 0                              # Flag: playing away
df_away['opponent_team_id'] = df_away['team_h']     # Opponent is the home team
df_away['opponent_difficulty'] = df_away['team_a_difficulty']  # Use away difficulty

# 4️⃣ Combine Home and Away Records
# Since merge operations can create different column sets, find common columns
cols_to_keep = list(set(df_home.columns).intersection(set(df_away.columns)))
# Concatenate home and away player records into one unified dataset
df_merged = pd.concat([df_home[cols_to_keep], df_away[cols_to_keep]], ignore_index=True)

# 5️⃣ Add Team Strength Metrics
# Enrich with team-level strength ratings (attack, defense, home/away)
df_merged = df_merged.merge(
    teams_df,
    left_on='team',               # Match on team ID
    right_on='id',                # Team ID from teams dataframe
    suffixes=('', '_team'),       # Avoid column name conflicts
    how='left'                    # Keep all records
)

# Clean up duplicate 'id' column created by the merge
if 'id_team' in df_merged.columns:
    df_merged.drop(columns=['id_team'], inplace=True)

# 6️⃣ Verify Final Dataset
print("\n" + "="*60)
print("FINAL MERGED DATASET")
print("="*60)
print(f"Shape: {df_merged.shape[0]} rows × {df_merged.shape[1]} columns")
print("\nSample records:")
print(df_merged[['id', 'gw', 'team', 'opponent_team_id', 'is_home', 'opponent_difficulty']].head(10))
print("\nAll available columns:")
print(df_merged.columns.tolist())

Columns after merging player info: Index(['minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded',
       'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards',
       'red_cards', 'saves', 'bonus', 'bps', 'influence_x', 'creativity_x',
       'threat_x', 'ict_index_x', 'clearances_blocks_interceptions',
       'recoveries', 'tackles', 'defensive_contribution', 'starts',
       'expected_goals', 'expected_assists', 'expected_goal_involvements',
       'expected_goals_conceded', 'total_points', 'in_dreamteam', 'id', 'gw',
       'first_name', 'second_name', 'team', 'element_type', 'now_cost',
       'chance_of_playing_next_round', 'form', 'influence_y', 'creativity_y',
       'threat_y', 'ict_index_y'],
      dtype='object')
Merged dataset shape: (17636, 60)
   id  gw  team  opponent_team_id  is_home  opponent_difficulty
0   1   1     1               NaN        1                  NaN
1   1   1     1               NaN        1                  NaN
2   1   1   

## Summary: Final Dataset Structure

The merged dataset (`df_merged`) now contains a comprehensive view of each player's performance in every gameweek with full context:

### **Performance Metrics** (from gameweek stats)
- Minutes played, goals, assists, bonus points
- Expected goals (xG), expected assists (xA)
- Clean sheets, saves, penalties, cards
- And many more...

### **Player Attributes** (from bootstrap data)
- Name, team, position, cost
- Form, ICT index (influence, creativity, threat)
- Injury/availability status

### **Match Context** (from fixtures)
- Home vs away status
- Opponent team
- Fixture difficulty rating
- Kickoff time

### **Team Strength** (from bootstrap data)
- Attack/defense ratings
- Home/away performance metrics
- Team form